1. IMPORTAR LIBRERÍAS

In [18]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from mlflow.models.signature import infer_signature

import mlflow
import mlflow.sklearn

2. CONFIGURAR MLFLOW

In [19]:
mlflow.set_tracking_uri("http://127.0.0.1:9090")

mlflow.set_experiment(
    "estudiantes_arbol_practica"
)

2026/05/16 23:49:17 INFO mlflow.tracking.fluent: Experiment with name 'estudiantes_arbol_practica' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/5', creation_time=1778993357181, experiment_id='5', last_update_time=1778993357181, lifecycle_stage='active', name='estudiantes_arbol_practica', tags={}, trace_location=None, workspace='default'>

3. LEER DATASET

In [20]:
df = pd.read_csv("data/estudiantes.csv")

print(df.head())

print(df.shape)

print(df.info())

        carrera   modalidad beca  edad  promedio  asistencias aprobado
0    Industrial  Presencial   Si    29       5.8           64       Si
1    Industrial     Hibrida   Si    27       6.6           51       No
2  Arquitectura  Presencial   Si    29       8.2           84       Si
3      Economia  Presencial   Si    29       6.6           67       No
4      Economia  Presencial   Si    24       5.1           72       No
(5000, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   carrera      5000 non-null   object 
 1   modalidad    5000 non-null   object 
 2   beca         5000 non-null   object 
 3   edad         5000 non-null   int64  
 4   promedio     5000 non-null   float64
 5   asistencias  5000 non-null   int64  
 6   aprobado     5000 non-null   object 
dtypes: float64(1), int64(2), object(4)
memory usage: 273.6+ KB
None


4. PREPARAR VARIABLE OBJETIVO

In [21]:
df["aprobado"] = df["aprobado"].map({
    "Si": 1,
    "No": 0
})

5. DEFINIR X Y Y

In [22]:
X = df.drop(columns=["aprobado"])

y = df["aprobado"]

6. VARIABLES CATEGÓRICAS Y NUMÉRICAS

In [23]:
columnas_categoricas = X.select_dtypes(
    include=["object"]
).columns.tolist()

columnas_numericas = X.select_dtypes(
    exclude=["object"]
).columns.tolist()

print(columnas_categoricas)

print(columnas_numericas)

['carrera', 'modalidad', 'beca']
['edad', 'promedio', 'asistencias']


7. DIVIDIR TRAIN Y TEST

In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

8. PREPROCESAMIENTO

In [25]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            columnas_categoricas
        ),
        (
            "num",
            "passthrough",
            columnas_numericas
        )
    ]
)

9. LAS 10 ITERACIONES

In [26]:
iteraciones = [

    {
        "criterion": "gini",
        "max_depth": 3,
        "min_samples_split": 50,
        "min_samples_leaf": 20
    },

    {
        "criterion": "gini",
        "max_depth": 4,
        "min_samples_split": 80,
        "min_samples_leaf": 30
    },

    {
        "criterion": "gini",
        "max_depth": 5,
        "min_samples_split": 100,
        "min_samples_leaf": 50
    },

    {
        "criterion": "gini",
        "max_depth": 6,
        "min_samples_split": 120,
        "min_samples_leaf": 60
    },

    {
        "criterion": "entropy",
        "max_depth": 3,
        "min_samples_split": 50,
        "min_samples_leaf": 20
    },

    {
        "criterion": "entropy",
        "max_depth": 4,
        "min_samples_split": 80,
        "min_samples_leaf": 30
    },

    {
        "criterion": "entropy",
        "max_depth": 5,
        "min_samples_split": 100,
        "min_samples_leaf": 50
    },

    {
        "criterion": "entropy",
        "max_depth": 6,
        "min_samples_split": 120,
        "min_samples_leaf": 60
    },

    {
        "criterion": "log_loss",
        "max_depth": 5,
        "min_samples_split": 100,
        "min_samples_leaf": 50
    },

    {
        "criterion": "log_loss",
        "max_depth": 7,
        "min_samples_split": 150,
        "min_samples_leaf": 70
    }
]

10. ENTRENAMIENTO + MLFLOW

In [27]:
while mlflow.active_run() is not None:
    mlflow.end_run()
for i, params in enumerate(iteraciones, start=1):

    with mlflow.start_run(
        run_name=f"iteracion_{i}"
    ) as run:

        pipeline = Pipeline(
            steps=[

                (
                    "preprocessor",
                    preprocessor
                ),

                (
                    "modelo",

                    DecisionTreeClassifier(
                        criterion=params["criterion"],
                        max_depth=params["max_depth"],
                        min_samples_split=params["min_samples_split"],
                        min_samples_leaf=params["min_samples_leaf"],
                        random_state=42
                    )
                )
            ]
        )

        pipeline.fit(X_train, y_train)

        y_pred = pipeline.predict(X_test)

🏃 View run merciful-frog-210 at: http://127.0.0.1:9090/#/experiments/4/runs/79aee6d19e374a9c9c2bf1d61df1db6f
🧪 View experiment at: http://127.0.0.1:9090/#/experiments/4
🏃 View run iteracion_1 at: http://127.0.0.1:9090/#/experiments/5/runs/02d6c9defe1b4f28bd76e18e1a782dc7
🧪 View experiment at: http://127.0.0.1:9090/#/experiments/5
🏃 View run iteracion_2 at: http://127.0.0.1:9090/#/experiments/5/runs/4a3224f8e68e4e1081ece878f065c9c3
🧪 View experiment at: http://127.0.0.1:9090/#/experiments/5
🏃 View run iteracion_3 at: http://127.0.0.1:9090/#/experiments/5/runs/1c57d1d616a542b8b38bc660d84e681c
🧪 View experiment at: http://127.0.0.1:9090/#/experiments/5
🏃 View run iteracion_4 at: http://127.0.0.1:9090/#/experiments/5/runs/4635f0394c904d88b1bc94bbcbe847a3
🧪 View experiment at: http://127.0.0.1:9090/#/experiments/5
🏃 View run iteracion_5 at: http://127.0.0.1:9090/#/experiments/5/runs/106fc09aa7724c6ca0dea915570c2d13
🧪 View experiment at: http://127.0.0.1:9090/#/experiments/5
🏃 View run itera

11. MÉTRICAS

In [28]:
        accuracy = accuracy_score(
            y_test,
            y_pred
        )

        precision = precision_score(
            y_test,
            y_pred,
            zero_division=0
        )

        recall = recall_score(
            y_test,
            y_pred,
            zero_division=0
        )

        f1 = f1_score(
            y_test,
            y_pred,
            zero_division=0
        )

12. GUARDAR PARÁMETROS EN MLFLOW

In [29]:
        mlflow.log_param(
            "iteracion",
            i
        )

        mlflow.log_param(
            "criterion",
            params["criterion"]
        )

        mlflow.log_param(
            "max_depth",
            params["max_depth"]
        )

        mlflow.log_param(
            "min_samples_split",
            params["min_samples_split"]
        )

        mlflow.log_param(
            "min_samples_leaf",
            params["min_samples_leaf"]
        )

70

13. GUARDAR MÉTRICAS EN MLFLOW

In [30]:
        mlflow.log_metric(
            "accuracy",
            accuracy
        )

        mlflow.log_metric(
            "precision",
            precision
        )

        mlflow.log_metric(
            "recall",
            recall
        )

        mlflow.log_metric(
            "f1_score",
            f1
        )

14. GUARDAR MODELO

In [31]:
  input_example = X_train.head(5)

signature = infer_signature(
    input_example,
    pipeline.predict(input_example)
)

mlflow.sklearn.log_model(
    sk_model=pipeline,
    artifact_path=f"modelo_iteracion_{i}",
    registered_model_name="Modelo_tarea2",
    input_example=input_example,
    signature=signature
)

C:\Users\dvas_\OneDrive\Documents\herramientasia\clase_learn\entorno-ok\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/16 23:49:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/16 23:49:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or clo

Registered model 'Modelo_tarea2' already exists. Creating a new version of this model...
2026/05/16 23:49:21 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Modelo_tarea2, version 2
Created version '2' of model 'Modelo_tarea2'.


15. MOSTRAR RESULTADOS

In [32]:
        print(f"Iteración {i}")

        print("Accuracy:", accuracy)

        print("F1:", f1)

        print("-" * 50)

Iteración 10
Accuracy: 0.851
F1: 0.807741935483871
--------------------------------------------------
